# Amazon Bin Inventory Reconciliation - Colab runner
Runtime -> Change runtime type -> **T4 GPU**. Runs the whole pipeline and zips the results so you can commit them to the repo.

## 1. Get the code
The repo is private, so either upload the zip of the repo (Files panel) and unzip, or clone with a GitHub personal access token (fine-grained, read-only, this repo only). Never commit the token.

In [ ]:
# Option A: clone (replace USER and paste token when asked by getpass)
from getpass import getpass
import os
token = getpass('GitHub token: ')
!git clone https://{token}@github.com/USER/amazon-bin-reconciliation.git repo
%cd repo

In [ ]:
# Option B: uploaded zip
# !unzip -q amazon-bin-reconciliation.zip -d repo && %cd repo

In [ ]:
!pip install -q scikit-image tabulate gradio
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Sanity check (synthetic data, ~1 min)

In [ ]:
!python -m pytest -q tests

## 3. Data (~10-15 min for 20k images)

In [ ]:
!python -m src.download_data --n 20000 --resize 256 --workers 48
!python -m src.prepare_splits

## 4. Classical baselines (CPU, ~10-20 min)

In [ ]:
!python -m src.train_classical --max-train 6000 --cv 5

## 5. CNNs
LR range test first, then the training runs. Each run is roughly 1-2 min/epoch on a T4 at 20k images.

In [ ]:
!python -m src.train_cnn --arch resnet34 --optimizer sgd --lr-find

In [ ]:
!python -m src.train_cnn --arch resnet18 --optimizer sgd --scheduler step --lr 0.01 --epochs 10 --run-name resnet18_sgd_step
!python -m src.train_cnn --arch resnet34 --optimizer sgd --scheduler sgdr --lr 0.01 --epochs 12 --run-name resnet34_sgd_sgdr

In [ ]:
!python -m src.train_cnn --arch resnet34 --optimizer adamw --scheduler onecycle --lr 1e-3 --epochs 12 --class-weights --run-name resnet34_adamw_onecycle_cw
!python -m src.train_cnn --arch efficientnet_b0 --optimizer adamw --scheduler onecycle --lr 2e-3 --epochs 12 --run-name effnetb0_adamw_onecycle

## 6. Evaluate on the test split + summary

In [ ]:
import glob
for c in sorted(glob.glob('models/*.pt')):
    if 'smoke_test' in c: continue
    !python -m src.evaluate --ckpt {c} --tta
!python -m src.summarize_results
!cat results/summary.md

## 7. Save results (commit `results/` to GitHub; keep the best .pt for the demo)

In [ ]:
!zip -qr results_and_models.zip results models
from google.colab import files
files.download('results_and_models.zip')

## 8. (Optional) Live demo from Colab

In [ ]:
!python app.py --ckpt models/resnet34_sgd_sgdr.pt --share